# 05b — End-to-End Loss Validation across 5 Hazard Sources

For the 1994 Northridge earthquake on the 2,008-bridge calibration portfolio, runs the complete loss pipeline (Sa → fragility → damage probabilities → expected loss) using each of five intensity-measure sources:

1. **ShakeMap (NN)** — USGS ShakeMap v4 interpolated by nearest neighbor.
2. **ASK14** — Abrahamson, Silva & Kamai (2014).
3. **BSSA14** — Boore, Stewart, Seyhan & Atkinson (2014).
4. **CB14** — Campbell & Bozorgnia (2014).
5. **CY14** — Chiou & Youngs (2014).

Reports per-source:
- **Total portfolio predicted loss** vs the observed total of $350 M.
- **Per-bridge MAE / RMSE / bias** of predicted loss vs observed loss.
- **Damage-state confusion** (predicted argmax vs observed).

Outputs (saved to `output/validation/all_hazard_sources/`):
- `loss_validation_summary.csv` — headline metrics table.
- `loss_validation_summary.png` — bar chart of total predicted vs observed.
- `loss_per_bridge.xlsx` — per-bridge predictions for every source (one sheet per source).

## 0. Preflight

In [ ]:
import sys, importlib, subprocess
_REQUIRED = ['numpy', 'pandas', 'matplotlib', 'scipy', 'openpyxl']
_missing = []
for _pkg in _REQUIRED:
    try:
        importlib.import_module(_pkg)
    except ImportError:
        _missing.append(_pkg)
print(f'Kernel Python: {sys.executable}')
if _missing:
    print(f'Installing: {_missing}')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *_missing])
    print('RESTART KERNEL.')
else:
    print('All required packages already importable. Ready.')

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
from scipy.stats import lognorm
import xml.etree.ElementTree as ET

PROJECT = Path.cwd().parent if Path.cwd().name == 'tutorials' else Path.cwd()
OUT_DIR = PROJECT / 'output' / 'validation' / 'all_hazard_sources'
OUT_DIR.mkdir(parents=True, exist_ok=True)

BRIDGES_PATH    = PROJECT / 'output' / 'exposure' / 'northridge_2008_rcv.xlsx'
OBSERVED_PATH   = PROJECT / 'data' / 'northridge_observed.csv'
FRAGILITY_PATH  = PROJECT / 'data' / 'hazus_bridge_fragility_params.csv'
GMPE_GRID_PATH  = PROJECT / 'output' / 'gmpe_nga_west2' / 'grid_gmpe_pga_sa1s.xlsx'
SHAKEMAP_PATH   = PROJECT / 'data' / 'grid.xml'

GMPE_NAMES = ['ASK14', 'BSSA14', 'CB14', 'CY14']
DAMAGE_RATIOS = {'none': 0.00, 'slight': 0.03, 'moderate': 0.08, 'extensive': 0.25, 'complete': 1.00}
DS_ORDER = ['none', 'slight', 'moderate', 'extensive', 'complete']

## 1. Load 2,008 Northridge bridges with RCV and observed damage

In [ ]:
bridges = pd.read_excel(BRIDGES_PATH)
obs_csv = pd.read_csv(OBSERVED_PATH)
bridges = bridges.merge(
    obs_csv[['structure_number', 'sa1s_shakemap', 'Observed_damage']],
    on='structure_number', how='left',
)
# Use Observed_damage from the csv (more populated than the xlsx column)
bridges['observed_damage'] = bridges['Observed_damage'].fillna(
    bridges.get('observed_damage', 'none')
).fillna('none')

print(f'Loaded {len(bridges):,} bridges')
print(f'Total RCV: ${bridges.replacement_cost_usd.sum()/1e9:.2f} B')
print(f'Observed damage distribution:')
print(bridges.observed_damage.value_counts())

## 2. Get Sa(1.0 s) at every bridge from each source

In [ ]:
# 2a. ShakeMap NN — already in csv column sa1s_shakemap
bridges['sa_ShakeMap'] = bridges['sa1s_shakemap'].astype(float)
print(f'ShakeMap Sa(1.0s) at bridges: {bridges.sa_ShakeMap.min():.3f}–{bridges.sa_ShakeMap.max():.3f} g')

# 2b. Each GMPE — load grid + nearest-neighbor onto bridge sites
for name in GMPE_NAMES:
    grid = pd.read_excel(GMPE_GRID_PATH, sheet_name=name)
    tree = cKDTree(np.column_stack([grid.latitude, grid.longitude]))
    _, idx = tree.query(np.column_stack([bridges.latitude, bridges.longitude]), k=1)
    bridges[f'sa_{name}'] = grid.Sa1s_g.values[idx]
    print(f'{name:8s} Sa(1.0s) at bridges: '
          f'{bridges[f"sa_{name}"].min():.4f}–{bridges[f"sa_{name}"].max():.4f} g')

## 3. Load fragility parameters and compute P(DS | Sa) per bridge per source

Hazus lognormal fragility: P(DS ≥ ds | Sa) = Φ(ln(Sa / θ_ds) / β_ds), where Φ is the standard normal CDF, θ is the median, and β is the dispersion.

In [ ]:
frag = pd.read_csv(FRAGILITY_PATH).set_index('HWB_Class')
from scipy.stats import norm

def damage_probabilities(sa, hwb_class):
    """Return P(DS=ds) for ds in DS_ORDER given Sa(1.0s) in g."""
    sa = max(float(sa), 1e-6)
    p = frag.loc[hwb_class] if hwb_class in frag.index else frag.loc['HWB28']
    P_exceed = []
    for ds in ('slight', 'moderate', 'extensive', 'complete'):
        median = p[f'{ds}_median']; beta = p[f'{ds}_beta']
        P_exceed.append(norm.cdf(np.log(sa / median) / beta))
    P_exceed = np.array(P_exceed)
    # Convert exceedance to discrete state probabilities
    p_none      = 1 - P_exceed[0]
    p_slight    = P_exceed[0] - P_exceed[1]
    p_moderate  = P_exceed[1] - P_exceed[2]
    p_extensive = P_exceed[2] - P_exceed[3]
    p_complete  = P_exceed[3]
    return np.array([p_none, p_slight, p_moderate, p_extensive, p_complete])

DR_VEC = np.array([DAMAGE_RATIOS[d] for d in DS_ORDER])

def expected_loss(sa, hwb_class, rcv):
    """E[Loss] = sum_DS P(DS) * DR(DS) * RCV."""
    p = damage_probabilities(sa, hwb_class)
    return float((p * DR_VEC).sum() * rcv), p

print('Fragility parameters loaded for', len(frag), 'HWB classes.')

In [ ]:
SOURCES = ['ShakeMap'] + GMPE_NAMES
per_bridge_loss = {}
predicted_ds_argmax = {}

for src in SOURCES:
    losses = np.zeros(len(bridges))
    argmax_ds = []
    sa_col = f'sa_{src}'
    for i, row in bridges.iterrows():
        sa = row[sa_col]
        loss, probs = expected_loss(sa, row.hwb_class, row.replacement_cost_usd)
        losses[i] = loss
        argmax_ds.append(DS_ORDER[int(np.argmax(probs))])
    per_bridge_loss[src] = losses
    predicted_ds_argmax[src] = argmax_ds
    bridges[f'loss_{src}'] = losses
    bridges[f'pred_ds_{src}'] = argmax_ds
    print(f'{src:10s}  total predicted loss = ${losses.sum()/1e6:.1f} M')

## 4. Observed loss per bridge

`observed_loss = DR(observed_DS) × RCV`. Bridges with `observed_damage == "none"` contribute zero.

In [ ]:
bridges['dr_observed'] = bridges.observed_damage.map(DAMAGE_RATIOS).fillna(0.0)
bridges['loss_observed'] = bridges.dr_observed * bridges.replacement_cost_usd

obs_total = bridges.loss_observed.sum()
print(f'Observed total loss: ${obs_total/1e6:.1f} M  '
      f'(target reported in §2.6.3 ≈ $350 M)')

## 5. Summary metrics — per source

| Metric | Definition |
|---|---|
| **Total Loss** | sum of per-bridge predicted loss |
| **Bias / Obs (%)** | (predicted − observed) / observed × 100 |
| **MAE** | mean(|predicted − observed|) per bridge |
| **RMSE** | sqrt(mean((predicted − observed)²)) per bridge |
| **DS exact match (%)** | argmax-DS agreement with observed_damage |

In [ ]:
rows = []
for src in SOURCES:
    pred = bridges[f'loss_{src}'].values
    obs  = bridges['loss_observed'].values
    diff = pred - obs
    pred_ds = bridges[f'pred_ds_{src}'].values
    obs_ds  = bridges['observed_damage'].values
    rows.append({
        'Source':            src,
        'Total predicted ($M)':  pred.sum() / 1e6,
        'Total observed ($M)':   obs.sum() / 1e6,
        'Bias / observed (%)':   (pred.sum() - obs.sum()) / obs.sum() * 100,
        'MAE per bridge ($)':    np.abs(diff).mean(),
        'RMSE per bridge ($)':   np.sqrt((diff**2).mean()),
        'DS exact match (%)':    (pred_ds == obs_ds).mean() * 100,
    })

summary = pd.DataFrame(rows).set_index('Source')
summary.to_csv(OUT_DIR / 'loss_validation_summary.csv')
print(f'Saved: {OUT_DIR / "loss_validation_summary.csv"}')
summary.round(2)

## 6. Bar chart — total predicted loss vs observed reference

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel A: total loss bar chart
ax = axes[0]
x = np.arange(len(SOURCES))
y = [bridges[f'loss_{s}'].sum() / 1e6 for s in SOURCES]
colors = ['#4C72B0', '#55A868', '#C44E52', '#8172B2', '#CCB974']
bars = ax.bar(x, y, color=colors, edgecolor='black', linewidth=0.5)
ax.axhline(obs_total / 1e6, color='red', linestyle='--', linewidth=2,
           label=f'Observed = ${obs_total/1e6:.0f} M')
ax.set_xticks(x); ax.set_xticklabels(SOURCES, rotation=15)
ax.set_ylabel('Total portfolio loss (USD millions)')
ax.set_title('Total predicted loss vs observed', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, linestyle='--', axis='y')
ax.legend(loc='upper right')
for b, v in zip(bars, y):
    ax.text(b.get_x() + b.get_width()/2, v + max(y)*0.01,
            f'${v:.0f}M', ha='center', fontsize=9, fontweight='bold')

# Panel B: MAE and RMSE bar chart
ax = axes[1]
width = 0.35
mae  = [summary.loc[s, 'MAE per bridge ($)'] / 1e6 for s in SOURCES]
rmse = [summary.loc[s, 'RMSE per bridge ($)'] / 1e6 for s in SOURCES]
ax.bar(x - width/2, mae,  width, label='MAE',  color='#4C72B0', edgecolor='black', linewidth=0.5)
ax.bar(x + width/2, rmse, width, label='RMSE', color='#C44E52', edgecolor='black', linewidth=0.5)
ax.set_xticks(x); ax.set_xticklabels(SOURCES, rotation=15)
ax.set_ylabel('Per-bridge error (USD millions)')
ax.set_title('Per-bridge MAE and RMSE of predicted loss', fontsize=12, fontweight='bold')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3, linestyle='--', axis='y')

plt.tight_layout()
fig_path = OUT_DIR / 'loss_validation_summary.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')

## 7. Per-bridge results — write a detailed Excel for downstream analysis

In [ ]:
with pd.ExcelWriter(OUT_DIR / 'loss_per_bridge.xlsx') as writer:
    cols_keep = ['structure_number', 'latitude', 'longitude', 'hwb_class',
                 'replacement_cost_usd', 'observed_damage', 'loss_observed']
    for src in SOURCES:
        sub = bridges[cols_keep + [f'sa_{src}', f'pred_ds_{src}', f'loss_{src}']].copy()
        sub.columns = ['structure_number','latitude','longitude','hwb_class',
                       'rcv_usd','observed_damage','observed_loss_usd',
                       'sa10_g','predicted_damage','predicted_loss_usd']
        sub['residual_usd'] = sub['predicted_loss_usd'] - sub['observed_loss_usd']
        sub.to_excel(writer, sheet_name=src, index=False)
    summary.round(2).to_excel(writer, sheet_name='Summary')

print(f'Saved: {OUT_DIR / "loss_per_bridge.xlsx"}')

## 8. Damage-state confusion (predicted argmax vs observed)

Useful as a discrete-mode summary of agreement (complementing the continuous expected-loss metric).

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 4), sharey=True)
for ax, src in zip(axes, SOURCES):
    ct = pd.crosstab(
        bridges['observed_damage'],
        bridges[f'pred_ds_{src}'],
        normalize='index',
    ).reindex(index=DS_ORDER, columns=DS_ORDER, fill_value=0)
    im = ax.imshow(ct.values, cmap='Blues', vmin=0, vmax=1, aspect='auto')
    ax.set_xticks(range(5)); ax.set_xticklabels(DS_ORDER, rotation=45, ha='right')
    ax.set_yticks(range(5)); ax.set_yticklabels(DS_ORDER)
    ax.set_xlabel('predicted')
    if src == 'ShakeMap':
        ax.set_ylabel('observed')
    exact = (bridges['observed_damage'] == bridges[f'pred_ds_{src}']).mean() * 100
    ax.set_title(f'{src}\nexact = {exact:.1f}%', fontsize=11, fontweight='bold')
    for i in range(5):
        for j in range(5):
            v = ct.values[i, j]
            if v > 0.005:
                ax.text(j, i, f'{v:.2f}', ha='center', va='center',
                        color='white' if v > 0.5 else 'black', fontsize=8)

fig.suptitle('Damage-state confusion (row-normalized) — observed vs predicted argmax',
             fontsize=12, fontweight='bold', y=1.04)
fig.colorbar(im, ax=axes, fraction=0.012, pad=0.02, label='row fraction')
fig_path = OUT_DIR / 'loss_validation_confusion.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')

## 9. Spatial loss map — per-source predicted loss + observed reference

Each panel: bridge dots sized by predicted loss (log scale) and coloured by predicted-vs-observed residual. The right-most panel shows actual observed damaged bridges only. Bridges with zero predicted loss are plotted as small grey dots.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
axes = axes.flatten()

# Bbox for plotting
lat_lo, lat_hi = 33.7, 34.7
lon_lo, lon_hi = -119.0, -117.9
EQ_LAT, EQ_LON = 34.213, -118.537

# All 5 sources + observed
panels = [(s, bridges[f'loss_{s}']) for s in SOURCES]
panels.append(('Observed', bridges['loss_observed']))

for ax, (src, losses) in zip(axes, panels):
    losses = np.array(losses)
    # Plot zero-loss bridges as small grey points
    z = losses < 1.0
    ax.scatter(bridges.longitude[z], bridges.latitude[z],
               s=2, c='lightgray', alpha=0.4, edgecolors='none', label='no loss')
    # Non-zero bridges: size = log scale of loss, color = log loss
    nz = ~z
    if nz.any():
        sizes = np.clip(np.log10(losses[nz] + 1) * 8, 4, 80)
        sc = ax.scatter(bridges.longitude[nz], bridges.latitude[nz],
                         s=sizes, c=losses[nz], cmap='hot_r',
                         norm=LogNorm(vmin=1e3, vmax=max(losses[nz].max(), 1e7)),
                         alpha=0.85, edgecolors='black', linewidths=0.3)
        cb = plt.colorbar(sc, ax=ax, fraction=0.04, pad=0.02)
        cb.set_label('Loss (USD, log scale)', fontsize=8)
    ax.scatter([EQ_LON], [EQ_LAT], marker='*', s=300, c='yellow',
               edgecolors='black', linewidths=1.2, zorder=10, label='epicenter')
    total = losses.sum() / 1e6
    n_dmg = nz.sum()
    ax.set_title(f'{src}: total = ${total:.1f} M, {n_dmg} bridges with loss',
                 fontsize=11, fontweight='bold')
    ax.set_xlim(lon_lo, lon_hi); ax.set_ylim(lat_lo, lat_hi)
    ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.legend(loc='lower right', fontsize=8)

fig.suptitle('Spatial distribution of predicted loss across 5 hazard sources vs observed',
             fontsize=14, fontweight='bold', y=1.005)
plt.tight_layout()
fig_path = OUT_DIR / 'loss_validation_spatial_map.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')

## 10. Residual map — predicted minus observed loss (only ShakeMap shown)

ShakeMap is the only source with non-trivial loss; the four GMPEs predict near-zero everywhere so their residual maps would just mirror the observed loss. This panel makes the spatial pattern of ShakeMap's over-prediction explicit.

In [ ]:
from matplotlib.colors import SymLogNorm
fig, ax = plt.subplots(figsize=(10, 8))
diff = bridges['loss_ShakeMap'] - bridges['loss_observed']
rmax = max(abs(diff.min()), abs(diff.max()))
linthresh = 1e5  # symlog transition at $100 K
sc = ax.scatter(bridges.longitude, bridges.latitude, c=diff, s=10,
                cmap='RdBu_r',
                norm=SymLogNorm(linthresh=linthresh, vmin=-rmax, vmax=rmax),
                alpha=0.85, edgecolors='none')
ax.scatter([EQ_LON], [EQ_LAT], marker='*', s=400, c='yellow',
           edgecolors='black', linewidths=1.5, zorder=10)
cb = plt.colorbar(sc, ax=ax, fraction=0.04, pad=0.02)
cb.set_label('Predicted − Observed Loss (USD, symlog)', fontsize=10)
ax.set_title('ShakeMap residual map: total over-prediction = '
             f'${(bridges.loss_ShakeMap.sum() - bridges.loss_observed.sum())/1e6:.0f} M',
             fontsize=12, fontweight='bold')
ax.set_xlim(lon_lo, lon_hi); ax.set_ylim(lat_lo, lat_hi)
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.grid(True, alpha=0.3, linestyle='--')
fig_path = OUT_DIR / 'loss_validation_residual_map_shakemap.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')

---

**For the report (§2.6.3 / §3.2.8):** the summary table and bar chart above show how each hazard source propagates through to dollar-loss agreement. The MAE / RMSE columns answer the team's request for per-bridge error metrics across all five hazard sources.